# FWI vs. Key Climate Drivers — MRI-ESM2-0 (Thailand)

**Analysis:** Annual area-weighted Fire Weather Index (FWI) against four climate drivers
(maximum temperature, precipitation, surface wind speed, and minimum relative humidity)
for Thailand (5.5°–20.5°N, 97.5°–105.5°E) across historical and nine SSP scenarios.

**Model:** MRI-ESM2-0 | **Source:** CMIP6  
**FWI dataset:** Dobrynin et al. (2021) pre-computed annual FWI (0.25° grid)  
**Driver datasets:** CMIP6 monthly atmospheric variables (~1.125° T160 grid, 160×320)

> **Note:** Multi-member scenarios (SSP126, SSP245, SSP370, SSP585) are concatenated
> along an `ensemble` dimension and reduced to the ensemble mean before plotting.  
> Single-member scenarios (SSP119, SSP434, SSP460, SSP534) have no ensemble dim.

> **Reproducibility:** Set `DATA_ROOT` in Section 2 to your local data root directory.


## 1  Dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from pathlib import Path
import glob as _glob
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy.stats import linregress


## 2  Configuration

Edit `DATA_ROOT` to point to your local data directory, then run all cells in order.


In [ ]:
# ── User configuration ───────────────────────────────────────────────────────
DATA_ROOT   = Path("/Volumes/jubjang/Jubjang")   # <-- change if needed
OUTPUT_FILE = "FWI_vs_4drivers_Thailand_2x2_MRI-ESM2-0_hursmin.png"

# Study region: Thailand
TH_SLICE = dict(lat=slice(5.5, 20.5), lon=slice(97.5, 105.5))

# Ensemble members used for historical and multi-member SSPs
# (r_index, i_index) pairs
VALID_COMBINATIONS = [
    (1, 1), (1, 2), (1, 1000),
    (2, 1), (3, 1), (4, 1), (5, 1),
    (6, 1), (7, 1), (8, 1), (9, 1), (10, 1),
]
VALID_COMBINATIONS_SSP585 = [
    (1, 1), (1, 2),
    (2, 1), (3, 1), (4, 1), (5, 1),
]

# Scenario keys and display labels
SCENARIOS = [
    "historical", "ssp119", "ssp126", "ssp245",
    "ssp370", "ssp460", "ssp534", "ssp585",
]
SCENARIO_LABELS = {
    "historical": "Historical",
    "ssp119":     "SSP1-1.9",
    "ssp126":     "SSP1-2.6",
    "ssp245":     "SSP2-4.5",
    "ssp370":     "SSP3-7.0",
    "ssp460":     "SSP4-6.0",
    "ssp534":     "SSP5-3.4",
    "ssp585":     "SSP5-8.5",
}

# Colour-blind-friendly palette (Wong, 2011 + extensions)
PALETTE = {
    "historical": "#000000",
    "ssp119":     "#E69F00",
    "ssp126":     "#56B4E9",
    "ssp245":     "#009E73",
    "ssp370":     "#882255",
    "ssp460":     "#D55E00",
    "ssp534":     "#0072B2",
    "ssp585":     "#CC79A7",
}

# cftime decoder (required for MRI-ESM2-0 non-Gregorian calendars)
TIME_CODER = xr.coding.times.CFDatetimeCoder(use_cftime=True)


## 3  Load datasets

### Grid resolutions

| Variable | Grid | Shape |
|---|---|---|
| `fwisa` | 0.25° × 0.25° (annual) | 72 × 144 |
| `tasmax`, `pr`, `sfcWind`, `hursmin` | T160 (~1.125°) | 160 × 320 |

### Ensemble handling

Multi-member scenarios are concatenated along an `ensemble` dimension.
Single-member scenarios (SSP119, SSP434, SSP460, SSP534) have no ensemble dim.


In [ ]:
def _open_glob(pattern, combine="nested", concat_dim="time", **kwargs):
    files = sorted(glob.glob(str(pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {pattern}")
    
    return xr.open_mfdataset(
        files, 
        combine=combine, 
        concat_dim=concat_dim, 
        engine="netcdf4", 
        decode_times=TIME_CODER,
        coords="minimal", 
        compat="override", 
        **kwargs
    )

def _open_multi_member(member_file_lists, **kwargs):
    members = []
    for i, pattern in enumerate(member_file_lists):
        files = sorted(glob.glob(str(pattern)))
        if not files:
            raise FileNotFoundError(f"No files for member {i}: {pattern}")
        
        ds = xr.open_mfdataset(
            files, 
            combine="by_coords", 
            engine="netcdf4",
            decode_times=TIME_CODER,
            **kwargs
        )
        members.append(ds.assign_coords(ensemble=i))
    return xr.concat(members, dim="ensemble")


### 3a  FWI (annual, 0.25° grid)

In [ ]:
FWI_DIR = DATA_ROOT / "FWICMIP6"

ds_fwi = {
    "historical": _open_multi_member([
        FWI_DIR / f"fwisa_ann_MRI-ESM2-0_historical_r{r}i{i}p1f1_g025.nc"
        for r, i in VALID_COMBINATIONS
    ]),
    "ssp119": _open_glob(FWI_DIR / "fwisa_ann_MRI-ESM2-0_ssp119_r4i1p1f1_g025.nc"),
    "ssp126": _open_multi_member([
        FWI_DIR / f"fwisa_ann_MRI-ESM2-0_ssp126_r{r}i1p1f1_g025.nc"
        for r in range(1, 6)
    ]),
    "ssp245": _open_multi_member([
        FWI_DIR / f"fwisa_ann_MRI-ESM2-0_ssp245_r{r}i1p1f1_g025.nc"
        for r in range(1, 6)
    ]),
    "ssp370": _open_multi_member([
        FWI_DIR / f"fwisa_ann_MRI-ESM2-0_ssp370_r{r}i1p1f1_g025.nc"
        for r in range(1, 6)
    ]),
    "ssp434": _open_glob(FWI_DIR / "fwisa_ann_MRI-ESM2-0_ssp434_r1i1p1f1_g025.nc"),
    "ssp460": _open_glob(FWI_DIR / "fwisa_ann_MRI-ESM2-0_ssp460_r1i1p1f1_g025.nc"),
    "ssp534": _open_glob(FWI_DIR / "fwisa_ann_MRI-ESM2-0_ssp534-over_r1i1p1f1_g025.nc"),
    "ssp585": _open_multi_member([
        FWI_DIR / f"fwisa_ann_MRI-ESM2-0_ssp585_r{r}i{i}p1f1_g025.nc"
        for r, i in VALID_COMBINATIONS_SSP585
    ]),
}

print("FWI datasets loaded successfully.")

### 3b  tasmax (monthly, T160 grid)

In [ ]:
TAS_DIR = DATA_ROOT / "tasmax"

ds_tasmax = {
    "historical": _open_multi_member([
        TAS_DIR / f"tasmax_Amon_MRI-ESM2-0_historical_r{r}i{i}p1f1_gn_185001-201412.nc"
        for r, i in VALID_COMBINATIONS
    ]),
    "ssp119": _open_glob(TAS_DIR / "tasmax_Amon_MRI-ESM2-0_ssp119_r4i1p1f1_gn_201501-210012.nc"),
    "ssp126": _open_multi_member([
        TAS_DIR / f"tasmax_Amon_MRI-ESM2-0_ssp126_r{r}i1p1f1_gn_*.nc"
        for r in range(1, 6)
    ]),
    "ssp245": _open_multi_member([
        TAS_DIR / f"tasmax_Amon_MRI-ESM2-0_ssp245_r{r}i1p1f1_gn*.nc"
        for r in range(1, 6)
    ]),
    "ssp370": _open_multi_member([
        TAS_DIR / f"tasmax_Amon_MRI-ESM2-0_ssp370_r{r}i1p1f1_gn_*.nc"
        for r in range(1, 6)
    ]),
    "ssp434": _open_glob(TAS_DIR / "tasmax_Amon_MRI-ESM2-0_ssp434_r1i1p1f1_gn*.nc",
                         decode_times=TIME_CODER, chunks={"time": 100}),
    "ssp460": _open_glob(TAS_DIR / "tasmax_Amon_MRI-ESM2-0_ssp460_r1i1p1f1_gn*.nc",
                         decode_times=TIME_CODER, chunks={"time": 100}),
    "ssp534": _open_glob(TAS_DIR / "tasmax_Amon_MRI-ESM2-0_ssp534-over_r*i1p1f1_gn*.nc",
                         decode_times=TIME_CODER, chunks={"time": 100}),
    "ssp585": _open_multi_member(
        [TAS_DIR / f"tasmax_Amon_MRI-ESM2-0_ssp585_r{r}i{i}p1f1_gn*.nc"
         for r, i in VALID_COMBINATIONS_SSP585]
    ),
}
print("tasmax datasets loaded.")


### 3c  pr (monthly, T160 grid)

In [ ]:
PR_DIR = DATA_ROOT / "pr"

ds_pr = {
    "historical": _open_multi_member([
        PR_DIR / f"pr_Amon_MRI-ESM2-0_historical_r{r}i{i}p1f1_gn_185001-201412.nc"
        for r, i in VALID_COMBINATIONS
    ]),
    "ssp119": _open_glob(PR_DIR / "pr_Amon_MRI-ESM2-0_ssp119_r4i1p1f1_gn_201501-210012.nc"),
    "ssp126": _open_multi_member([
        PR_DIR / f"pr_Amon_MRI-ESM2-0_ssp126_r{r}i1p1f1_gn_*.nc"
        for r in range(1, 6)
    ]),
    "ssp245": _open_multi_member([
        PR_DIR / f"pr_Amon_MRI-ESM2-0_ssp245_r{r}i1p1f1_gn*.nc"
        for r in range(1, 6)
    ]),
    "ssp370": _open_multi_member([
        PR_DIR / f"pr_Amon_MRI-ESM2-0_ssp370_r{r}i1p1f1_gn_*.nc"
        for r in range(1, 6)
    ]),
    "ssp434": _open_glob(PR_DIR / "pr_Amon_MRI-ESM2-0_ssp434_r1i1p1f1_gn_*.nc"),
    "ssp460": _open_glob(PR_DIR / "pr_Amon_MRI-ESM2-0_ssp460_r1i1p1f1_gn_*.nc"),
    "ssp534": _open_glob(PR_DIR / "pr_Amon_MRI-ESM2-0_ssp534-over_r1i1p1f1_gn*.nc",
                         decode_times=TIME_CODER, chunks={"time": 100}),
    "ssp585": _open_multi_member(
        [PR_DIR / f"pr_Amon_MRI-ESM2-0_ssp585_r{r}i{i}p1f1_gn*.nc"
         for r, i in VALID_COMBINATIONS_SSP585]
    ),
}
print("pr datasets loaded.")


### 3d  sfcWind (monthly, T160 grid)

In [ ]:
WIND_DIR = DATA_ROOT / "sfcWind"

ds_wind = {
    "historical": _open_multi_member([
        WIND_DIR / f"sfcWind_Amon_MRI-ESM2-0_historical_r{r}i{i}p1f1_gn_185001-201412.nc"
        for r, i in VALID_COMBINATIONS
    ]),
    "ssp119": _open_glob(WIND_DIR / "sfcWind_Amon_MRI-ESM2-0_ssp119_r4i1p1f1_gn_201501-210012.nc"),
    "ssp126": _open_multi_member([
        WIND_DIR / f"sfcWind_Amon_MRI-ESM2-0_ssp126_r{r}i1p1f1_gn_*.nc"
        for r in range(1, 6)
    ]),
    "ssp245": _open_multi_member([
        WIND_DIR / f"sfcWind_Amon_MRI-ESM2-0_ssp245_r{r}i1p1f1_gn*.nc"
        for r in range(1, 6)
    ]),
    "ssp370": _open_multi_member([
        WIND_DIR / f"sfcWind_Amon_MRI-ESM2-0_ssp370_r{r}i1p1f1_gn_*.nc"
        for r in range(1, 6)
    ]),
    "ssp434": _open_glob(WIND_DIR / "sfcWind_Amon_MRI-ESM2-0_ssp434_r1i1p1f1_gn_*.nc"),
    "ssp460": _open_glob(WIND_DIR / "sfcWind_Amon_MRI-ESM2-0_ssp460_r1i1p1f1_gn_*.nc"),
    "ssp534": _open_glob(WIND_DIR / "sfcWind_Amon_MRI-ESM2-0_ssp534-over_r1i1p1f1_gn*.nc",
                         decode_times=TIME_CODER, chunks={"time": 100}),
    "ssp585": _open_multi_member(
        [WIND_DIR / f"sfcWind_Amon_MRI-ESM2-0_ssp585_r{r}i{i}p1f1_gn*.nc"
         for r, i in VALID_COMBINATIONS_SSP585]
    ),
}
print("sfcWind datasets loaded.")


### 3e  hursmin (daily, T160 grid)

In [ ]:
HURS_DIR = DATA_ROOT / "hursmin"

ds_hurs = {
    "historical": _open_multi_member([
        HURS_DIR / f"hursmin_day_MRI-ESM2-0_historical_r{r}i{i}p1f1_gn_185001-201412.nc"
        for r, i in VALID_COMBINATIONS
    ]),
    "ssp119": _open_glob(HURS_DIR / "hursmin_day_MRI-ESM2-0_ssp119_r4i1p1f1_gn_201501-210012.nc"),
    "ssp126": _open_multi_member([
        HURS_DIR / f"hursmin_day_MRI-ESM2-0_ssp126_r{r}i1p1f1_gn_*.nc"
        for r in range(1, 6)
    ]),
    "ssp245": _open_multi_member([
        HURS_DIR / f"hursmin_day_MRI-ESM2-0_ssp245_r{r}i1p1f1_gn*.nc"
        for r in range(1, 6)
    ]),
    "ssp370": _open_multi_member([
        HURS_DIR / f"hursmin_day_MRI-ESM2-0_ssp370_r{r}i1p1f1_gn_*.nc"
        for r in range(1, 6)
    ]),
    "ssp460": _open_glob(HURS_DIR / "hursmin_day_MRI-ESM2-0_ssp460_r1i1p1f1_gn*.nc",
                         decode_times=TIME_CODER, chunks={"time": 100}),
    "ssp534": _open_glob(HURS_DIR / "hursmin_day_MRI-ESM2-0_ssp534-over_r*i1p1f1_gn*.nc",
                         decode_times=TIME_CODER, chunks={"time": 100}),
    "ssp585": _open_multi_member(
        [HURS_DIR / f"hursmin_day_MRI-ESM2-0_ssp585_r{r}i{i}p1f1_gn*.nc"
         for r, i in VALID_COMBINATIONS_SSP585]
    ),
}
print("hursmin datasets loaded.")


## 4  Compute grid-cell area weights

Two area grids are needed because FWI and the driver variables live on different grids:

| Grid | Shape | Used for |
|---|---|---|
| `areacella` | 72 × 144 | FWI (0.25° grid) |
| `areacella_drv` | 160 × 320 | tasmax / pr / sfcWind / hurs (T160 grid) |

Grid-cell areas are derived from the spherical Earth formula:
$$A_i = R^2 \, \Delta\lambda \, (\sin\phi_2 - \sin\phi_1)$$


In [ ]:
def _compute_areacella(lat_coords: np.ndarray, lon_coords: np.ndarray) -> xr.Dataset:
    """
    Return an xr.Dataset with variable ``area`` (km²) on the supplied lat/lon grid.
    """
    R = 6_378_137.0  # WGS-84 mean radius (m)
    lat_rad = np.radians(lat_coords)
    lon_rad = np.radians(lon_coords)
    d_lon = np.diff(lon_rad).mean()
    d_lat = np.diff(lat_rad).mean()

    area = np.zeros((len(lat_rad), len(lon_rad)))
    for i, phi in enumerate(lat_rad):
        phi1 = phi - d_lat / 2
        phi2 = phi + d_lat / 2
        area[i, :] = R**2 * d_lon * (np.sin(phi2) - np.sin(phi1))

    return xr.Dataset({
        "area": xr.DataArray(
            area / 1e6,          # m² → km²
            dims=["lat", "lon"],
            coords={"lat": lat_coords, "lon": lon_coords},
            attrs={"units": "km2", "long_name": "grid-cell area"},
        )
    })


# FWI grid  (72 lat × 144 lon) — analytical (no model areacella file for 0.25° grid)
areacella = _compute_areacella(
    lat_coords=np.linspace(-88.75,  88.75,  72),
    lon_coords=np.linspace(  1.25, 358.75, 144),
)

# Driver grid — load actual MRI-ESM2-0 model areacella file (same for all SSPs)
_areacella_drv_file = DATA_ROOT / "pr" / "areacella_fx_MRI-ESM2-0_ssp370_r1i1p1f1_gn.nc"
_areacella_drv_raw  = xr.open_dataset(_areacella_drv_file)
# Rename 'areacella' variable to 'area' for consistency with helper functions
areacella_drv = _areacella_drv_raw.rename({"areacella": "area"})

print("areacella     →", areacella)
print("areacella_drv →", areacella_drv)

## 5  Analysis helper functions

In [ ]:
def _ensemble_mean(ds: xr.Dataset) -> xr.Dataset:
    """Reduce the ensemble dimension to its mean if present; no-op otherwise."""
    if "ensemble" in ds.dims:
        return ds.mean(dim="ensemble")
    return ds


def _annual_mean(ds: xr.Dataset) -> xr.Dataset:
    """Resample monthly data to annual means (year-start label, cftime-safe)."""
    return ds.resample(time="YS").mean()


def _drop_time_dim(da: xr.Dataset) -> xr.Dataset:
    """Remove time dimension from an area Dataset if accidentally present."""
    if "time" in da.dims:
        da = da.isel(time=0)
        da = da.drop_vars([v for v in da.coords if "time" in v], errors="ignore")
    return da  # fixed: was `return daz` (NameError)


def _sortby_lat(ds: xr.Dataset) -> xr.Dataset:
    """Ensure latitude is ascending. Model files are often stored N→S (descending)."""
    if "lat" in ds.dims and float(ds.lat[0]) > float(ds.lat[-1]):
        return ds.sortby("lat")
    return ds


def _area_weighted_mean(da: xr.DataArray, weights: xr.Dataset,
                        region: dict) -> xr.DataArray:
    """
    Area-weighted spatial mean of *da* clipped to *region*.

    Parameters
    ----------
    da      : DataArray to average (must have lat/lon dims)
    weights : Dataset with an ``area`` variable on a compatible grid
    region  : dict of slice kwargs defining the study region
    """
    da_reg = da.sel(**region)
    # Select nearest area weights, then reassign coords to exactly match da_reg.
    # Without assign_coords, xarray keeps the areacella's slightly different lat/lon
    # values, causing a coordinate mismatch that makes .weighted().mean() return NaN.
    w = (
        weights
        .sel(lat=da_reg.lat, lon=da_reg.lon, method="nearest")
        ["area"]
        .assign_coords(lat=da_reg.lat, lon=da_reg.lon)
    )
    return da_reg.weighted(w).mean(dim=["lat", "lon"])


## 6  Plotting function

`plot_fwi_vs_4vars_grid` produces a 2 × 2 panel figure showing area-weighted
annual FWI (ensemble mean) against each of the four climate drivers for all
nine scenarios. A linear regression line is overlaid on each scatter cloud.

**Ensemble handling inside the function:**  
Multi-member datasets are reduced to their ensemble mean before spatial averaging,
so all scenarios are treated uniformly regardless of ensemble size.


In [ ]:
def plot_fwi_vs_4vars_grid(
    ds_fwi:    dict,
    ds_tasmax: dict,
    ds_pr:     dict,
    ds_wind:   dict,
    ds_hurs:   dict,
    scenarios: list,
    labels:    dict,
    palette:   dict,
    area_fwi:  xr.Dataset,
    area_drv:  xr.Dataset,
    region:    dict = TH_SLICE,
    filename:  str  = OUTPUT_FILE,
) -> None:
    """
    2 × 2 scatter plot: annual area-weighted ensemble-mean FWI vs. four climate drivers.

    Parameters
    ----------
    ds_fwi, ds_tasmax, ds_pr, ds_wind, ds_hurs
        Dicts keyed by scenario string containing xarray Datasets.
        Multi-member datasets must have an ``ensemble`` dimension.
    scenarios : ordered list of scenario keys
    labels    : dict mapping scenario key → legend label
    palette   : dict mapping scenario key → hex colour
    area_fwi  : areacella Dataset for the FWI grid (72 × 144)
    area_drv  : areacella Dataset for the driver grids (160 × 320)
    region    : lat/lon slice dict defining the study region
    filename  : output filename (PNG, 300 dpi)
    """
    area_fwi = _drop_time_dim(area_fwi)
    area_drv = _drop_time_dim(area_drv)

    panels = [
        dict(key="tasmax",  ds=ds_tasmax, xlabel="Maximum Temperature (°C)",   label="a)"),
        dict(key="pr",      ds=ds_pr,     xlabel="Precipitation (mm day⁻¹)",   label="b)"),
        dict(key="sfcWind", ds=ds_wind,   xlabel="Surface Wind Speed (m s⁻¹)", label="c)"),
        dict(key="hursmin",    ds=ds_hurs,   xlabel="Minimum Relative Humidity (%)",       label="(d)"),
    ]

    fig, axes = plt.subplots(
        2, 2, figsize=(5.51, 4.3),
        constrained_layout=True, sharey="row",
    )
    axes = axes.flatten()
    legend_handles = {}

    for ax, panel in zip(axes, panels):
        var = panel["key"]
        ax.set_title(panel["label"], loc="left", fontsize=13)

        for scen in scenarios:
            # ── FWI: ensemble mean → sort lat → region slice → area mean ────
            fwi_ds   = _sortby_lat(_ensemble_mean(ds_fwi[scen]))
            fwi_reg  = fwi_ds.sel(**region)
            fwi_mean = _area_weighted_mean(fwi_reg["fwisa"], area_fwi, region)

            # ── Driver: ensemble mean → sort lat → region slice → annual mean → area mean
            drv_ds  = _sortby_lat(_ensemble_mean(panel["ds"][scen]))
            drv_reg = drv_ds.sel(**region)
            drv_ann = _annual_mean(drv_reg).sel(
                time=fwi_reg["time"], method="nearest"
            )

            if var == "tasmax":
                x_mean = (
                    _area_weighted_mean(drv_ann["tasmax"], area_drv, region)
                    - 273.15            # K → °C
                )
            elif var == "pr":
                x_mean = (
                    _area_weighted_mean(drv_ann["pr"], area_drv, region)
                    * 86_400.0          # kg m⁻² s⁻¹ → mm day⁻¹
                )
            elif var == "sfcWind":
                x_mean = _area_weighted_mean(drv_ann["sfcWind"], area_drv, region)
            elif var == "hursmin":
                x_mean = _area_weighted_mean(drv_ann["hursmin"],    area_drv, region)
            else:
                continue

            x_vals = np.asarray(x_mean.values)
            y_vals = np.asarray(fwi_mean.values)

            # Remove NaNs and check alignment
            mask = (~np.isnan(x_vals)) & (~np.isnan(y_vals))
            x_vals, y_vals = x_vals[mask], y_vals[mask]
            if x_vals.size == 0 or x_vals.size != y_vals.size:
                print(f"[skip] {scen} / {var}: size mismatch or all-NaN")
                continue

            color = palette.get(scen, "#000000")
            sc = ax.scatter(x_vals, y_vals, s=5, alpha=0.6, color=color)

            slope, intercept, *_ = linregress(x_vals, y_vals)
            x_fit = np.linspace(x_vals.min(), x_vals.max(), 100)
            ax.plot(x_fit, slope * x_fit + intercept, "-", lw=1.5, color=color)

            lbl = labels.get(scen, scen)
            if lbl not in legend_handles:
                legend_handles[lbl] = sc

        ax.set_xlabel(panel["xlabel"], fontsize=11)
        ax.set_ylim(0, 50)
        ax.grid(True, linestyle="solid", alpha=0.6)
        ax.tick_params(labelsize=10)

    # Y-axis labels: left column only
    for idx, ax in enumerate(axes):
        if idx % 2 == 0:
            ax.set_ylabel("FWI", fontsize=13)
        else:
            ax.set_ylabel(None)
            ax.tick_params(labelleft=False)

    fig.legend(
        legend_handles.values(),
        legend_handles.keys(),
        loc="upper center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=3, frameon=False, fontsize=10,
    )

    fig.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Figure saved → {filename}")
    plt.show()


## 7  Generate figure

In [ ]:
# ── Diagnostic: run this cell BEFORE the plot to see exactly where NaN comes from ──
import numpy as np

scen = "historical"

print("=== FWI ===")
fwi_ds_d = _sortby_lat(_ensemble_mean(ds_fwi[scen]))
print(f"lat range in file : {float(fwi_ds_d.lat.min()):.3f} → {float(fwi_ds_d.lat.max()):.3f}")
fwi_reg_d = fwi_ds_d.sel(**TH_SLICE)
print(f"lat after TH_SLICE: {float(fwi_reg_d.lat.min()):.3f} → {float(fwi_reg_d.lat.max()):.3f}  (n={fwi_reg_d.sizes['lat']})")
print(f"lon after TH_SLICE: {float(fwi_reg_d.lon.min()):.3f} → {float(fwi_reg_d.lon.max()):.3f}  (n={fwi_reg_d.sizes['lon']})")
arr = fwi_reg_d["fwisa"].values
print(f"fwisa: shape={arr.shape}, NaN={np.isnan(arr).sum()}/{arr.size}, sample={arr[0,:3,:3]}")

print("\n=== Driver tasmax ===")
drv_ds_d = _sortby_lat(_ensemble_mean(ds_tasmax[scen]))
print(f"lat range in file : {float(drv_ds_d.lat.min()):.3f} → {float(drv_ds_d.lat.max()):.3f}")
drv_reg_d = drv_ds_d.sel(**TH_SLICE)
print(f"lat after TH_SLICE: {float(drv_reg_d.lat.min()):.3f} → {float(drv_reg_d.lat.max()):.3f}  (n={drv_reg_d.sizes['lat']})")
print(f"lon after TH_SLICE: {float(drv_reg_d.lon.min()):.3f} → {float(drv_reg_d.lon.max()):.3f}  (n={drv_reg_d.sizes['lon']})")
arr2 = drv_reg_d["tasmax"].values
print(f"tasmax: shape={arr2.shape}, NaN={np.isnan(arr2).sum()}/{arr2.size}")

print("\n=== Annual driver aligned to FWI times ===")
drv_ann_d = _annual_mean(drv_reg_d).sel(time=fwi_reg_d["time"], method="nearest")
print(f"drv_ann sizes: {dict(drv_ann_d.sizes)}")
arr3 = drv_ann_d["tasmax"].values
print(f"tasmax annual: shape={arr3.shape}, NaN={np.isnan(arr3).sum()}/{arr3.size}")

print("\n=== Area-weighted means ===")
w_fwi = np.cos(np.radians(fwi_reg_d.lat))
fwi_m = fwi_reg_d["fwisa"].weighted(w_fwi).mean(dim=["lat", "lon"])
print(f"fwi_mean (cos-lat): {fwi_m.values[:4]}")

w_drv = np.cos(np.radians(drv_ann_d.lat))
x_m = drv_ann_d["tasmax"].weighted(w_drv).mean(dim=["lat", "lon"])
print(f"tasmax_mean (cos-lat): {x_m.values[:4]}")


In [ ]:
plot_fwi_vs_4vars_grid(
    ds_fwi    = ds_fwi,
    ds_tasmax = ds_tasmax,
    ds_pr     = ds_pr,
    ds_wind   = ds_wind,
    ds_hurs   = ds_hurs,
    scenarios = SCENARIOS,
    labels    = SCENARIO_LABELS,
    palette   = PALETTE,
    area_fwi  = areacella,
    area_drv  = areacella_drv,
    region    = TH_SLICE,
    filename  = OUTPUT_FILE,
)


## 8  Pearson correlation, slope & intercept (driver vs FWI)

Area-weighted annual mean per scenario, then Pearson r / OLS slope / intercept reported per scenario and for all scenarios combined.

In [ ]:
from scipy.stats import pearsonr, linregress
import pandas as pd

def pearson_tasmax_vs_fwi(ds_fwi, ds_tasmax, scenarios, labels, areacella, areacella_drv):
    """Pearson r, slope, intercept: area-weighted annual tasmax (°C) vs FWI for Thailand."""
    combined_tas, combined_fwi = [], []
    rows = []
    for scen in scenarios:
        label = labels[scen]
        fwi_ds   = _sortby_lat(_ensemble_mean(ds_fwi[scen]))
        fwi_mean = _area_weighted_mean(fwi_ds["fwisa"], areacella, TH_SLICE).values
        drv_ds   = _sortby_lat(_ensemble_mean(ds_tasmax[scen]))
        drv_ann  = drv_ds.resample(time="YS").mean()
        drv_ann  = drv_ann.sel(time=fwi_ds.sel(**TH_SLICE)["time"], method="nearest")
        tas_mean = (_area_weighted_mean(drv_ann["tasmax"], areacella_drv, TH_SLICE) - 273.15).values
        mask = ~np.isnan(tas_mean) & ~np.isnan(fwi_mean)
        tas_mean, fwi_mean = tas_mean[mask], fwi_mean[mask]
        if len(tas_mean) > 1:
            r, _ = pearsonr(tas_mean, fwi_mean)
            slope, intercept, *_ = linregress(tas_mean, fwi_mean)
            rows.append({"Scenario": label, "r": round(r, 4), "slope": round(slope, 4), "intercept": round(intercept, 4)})
            combined_tas.extend(tas_mean); combined_fwi.extend(fwi_mean)
    if combined_tas:
        r, _ = pearsonr(combined_tas, combined_fwi)
        slope, intercept, *_ = linregress(combined_tas, combined_fwi)
        rows.append({"Scenario": "ALL (combined)", "r": round(r, 4), "slope": round(slope, 4), "intercept": round(intercept, 4)})
    df = pd.DataFrame(rows)
    print("\n── tasmax (°C) vs FWI — Thailand ──")
    print(df.to_string(index=False))
    return df

df_tasmax = pearson_tasmax_vs_fwi(
    ds_fwi, ds_tasmax, SCENARIOS, SCENARIO_LABELS, areacella, areacella_drv)

In [ ]:
def pearson_pr_vs_fwi(ds_fwi, ds_pr, scenarios, labels, areacella, areacella_drv):
    """Pearson r, slope, intercept: area-weighted annual pr (mm/day) vs FWI for Thailand."""
    combined_pr, combined_fwi = [], []
    rows = []
    for scen in scenarios:
        label = labels[scen]
        fwi_ds  = _sortby_lat(_ensemble_mean(ds_fwi[scen]))
        fwi_mean = _area_weighted_mean(fwi_ds["fwisa"], areacella, TH_SLICE).values
        drv_ds  = _sortby_lat(_ensemble_mean(ds_pr[scen]))
        drv_ann = drv_ds.resample(time="YS").mean()
        drv_ann = drv_ann.sel(time=fwi_ds.sel(**TH_SLICE)["time"], method="nearest")
        pr_mean = (_area_weighted_mean(drv_ann["pr"], areacella_drv, TH_SLICE) * 86_400).values
        mask = ~np.isnan(pr_mean) & ~np.isnan(fwi_mean)
        pr_mean, fwi_mean = pr_mean[mask], fwi_mean[mask]
        if len(pr_mean) > 1:
            r, _ = pearsonr(pr_mean, fwi_mean)
            slope, intercept, *_ = linregress(pr_mean, fwi_mean)
            rows.append({"Scenario": label, "r": round(r, 4), "slope": round(slope, 4), "intercept": round(intercept, 4)})
            combined_pr.extend(pr_mean); combined_fwi.extend(fwi_mean)
    if combined_pr:
        r, _ = pearsonr(combined_pr, combined_fwi)
        slope, intercept, *_ = linregress(combined_pr, combined_fwi)
        rows.append({"Scenario": "ALL (combined)", "r": round(r, 4), "slope": round(slope, 4), "intercept": round(intercept, 4)})
    df = pd.DataFrame(rows)
    print("\n── pr (mm/day) vs FWI — Thailand ──")
    print(df.to_string(index=False))
    return df

df_pr = pearson_pr_vs_fwi(
    ds_fwi, ds_pr, SCENARIOS, SCENARIO_LABELS, areacella, areacella_drv)

In [ ]:
def pearson_sfcWind_vs_fwi(ds_fwi, ds_wind, scenarios, labels, areacella, areacella_drv):
    """Pearson r, slope, intercept: area-weighted annual sfcWind (m/s) vs FWI for Thailand."""
    combined_wind, combined_fwi = [], []
    rows = []
    for scen in scenarios:
        label = labels[scen]
        fwi_ds    = _sortby_lat(_ensemble_mean(ds_fwi[scen]))
        fwi_mean  = _area_weighted_mean(fwi_ds["fwisa"], areacella, TH_SLICE).values
        drv_ds    = _sortby_lat(_ensemble_mean(ds_wind[scen]))
        drv_ann   = drv_ds.resample(time="YS").mean()
        drv_ann   = drv_ann.sel(time=fwi_ds.sel(**TH_SLICE)["time"], method="nearest")
        wind_mean = _area_weighted_mean(drv_ann["sfcWind"], areacella_drv, TH_SLICE).values
        mask = ~np.isnan(wind_mean) & ~np.isnan(fwi_mean)
        wind_mean, fwi_mean = wind_mean[mask], fwi_mean[mask]
        if len(wind_mean) > 1:
            r, _ = pearsonr(wind_mean, fwi_mean)
            slope, intercept, *_ = linregress(wind_mean, fwi_mean)
            rows.append({"Scenario": label, "r": round(r, 4), "slope": round(slope, 4), "intercept": round(intercept, 4)})
            combined_wind.extend(wind_mean); combined_fwi.extend(fwi_mean)
    if combined_wind:
        r, _ = pearsonr(combined_wind, combined_fwi)
        slope, intercept, *_ = linregress(combined_wind, combined_fwi)
        rows.append({"Scenario": "ALL (combined)", "r": round(r, 4), "slope": round(slope, 4), "intercept": round(intercept, 4)})
    df = pd.DataFrame(rows)
    print("\n── sfcWind (m/s) vs FWI — Thailand ──")
    print(df.to_string(index=False))
    return df

df_wind = pearson_sfcWind_vs_fwi(
    ds_fwi, ds_wind, SCENARIOS, SCENARIO_LABELS, areacella, areacella_drv)

In [ ]:
def pearson_hurs_vs_fwi(ds_fwi, ds_hurs, scenarios, labels, areacella, areacella_drv):
    """Pearson r, slope, intercept: area-weighted annual hursmin (%) vs FWI for Thailand."""
    combined_hurs, combined_fwi = [], []
    rows = []
    for scen in scenarios:
        label = labels[scen]
        fwi_ds    = _sortby_lat(_ensemble_mean(ds_fwi[scen]))
        fwi_mean  = _area_weighted_mean(fwi_ds["fwisa"], areacella, TH_SLICE).values
        drv_ds    = _sortby_lat(_ensemble_mean(ds_hurs[scen]))
        drv_ann   = drv_ds.resample(time="YS").mean()
        drv_ann   = drv_ann.sel(time=fwi_ds.sel(**TH_SLICE)["time"], method="nearest")
        hurs_mean = _area_weighted_mean(drv_ann["hursmin"], areacella_drv, TH_SLICE).values
        mask = ~np.isnan(hurs_mean) & ~np.isnan(fwi_mean)
        hurs_mean, fwi_mean = hurs_mean[mask], fwi_mean[mask]
        if len(hurs_mean) > 1:
            r, _ = pearsonr(hurs_mean, fwi_mean)
            slope, intercept, *_ = linregress(hurs_mean, fwi_mean)
            rows.append({"Scenario": label, "r": round(r, 4), "slope": round(slope, 4), "intercept": round(intercept, 4)})
            combined_hurs.extend(hurs_mean); combined_fwi.extend(fwi_mean)
    if combined_hurs:
        r, _ = pearsonr(combined_hurs, combined_fwi)
        slope, intercept, *_ = linregress(combined_hurs, combined_fwi)
        rows.append({"Scenario": "ALL (combined)", "r": round(r, 4), "slope": round(slope, 4), "intercept": round(intercept, 4)})
    df = pd.DataFrame(rows)
    print("\n── hurs (%) vs FWI — Thailand ──")
    print(df.to_string(index=False))
    return df

df_hurs = pearson_hurs_vs_fwi(
    ds_fwi, ds_hurs, SCENARIOS, SCENARIO_LABELS, areacella, areacella_drv)